In [2]:
# Cell 1
import os
import time
import json
import torch
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from IPython.display import display, HTML

# Import our modular engine
from tisd_engine import chat_with_tisd

# Setup device
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

# Define paths
TEST_DATA_FILE = "../data/test_set.json"

ImportError: cannot import name 'is_flash_attn_greater_or_equal_2_10' from 'transformers.utils' (/opt/homebrew/Caskroom/miniforge/base/envs/tisd/lib/python3.11/site-packages/transformers/utils/__init__.py)

In [ ]:
# Cell 2
from sentence_transformers import SentenceTransformer, util

print(f"Loading Semantic Judge (all-MiniLM-L6-v2) to {device}...")
# We use this model ONLY for scoring, not for the chatbot itself.
eval_judge = SentenceTransformer('all-MiniLM-L6-v2', device=device)

def calculate_semantic_score(pred, target):
    """Calculates Cosine Similarity between vectors of two sentences."""
    if not pred or not target:
        return 0.0
    
    # Encode both sentences into 384-dimensional vectors
    embeddings = eval_judge.encode([pred, target], convert_to_tensor=True, show_progress_bar=False)
    
    # Compute Cosine Similarity (Result is -1 to 1, we treat it as 0 to 1 here)
    cosine_sim = util.cos_sim(embeddings[0], embeddings[1])
    
    return float(cosine_sim)

def calculate_jaccard(pred, target):
    """The OLD metric for comparison (Word Overlap)."""
    set1 = set(pred.lower().split())
    set2 = set(target.lower().split())
    if not set2: return 0.0
    return len(set1.intersection(set2)) / len(set1.union(set2))

In [ ]:
# Cell 3
print(f"Reading test set from {TEST_DATA_FILE}...")
with open(TEST_DATA_FILE, "r") as f:
    test_questions = json.load(f)

results = []

print(f"Starting Evaluation on {len(test_questions)} questions...")
for item in tqdm(test_questions):
    q = item["question"]
    gt = item["ground_truth"]
    
    # 1. Run the RAG Pipeline
    # We pass None for class_filter to do a broad search
    answer, context, latency = chat_with_tisd(q)
    
    # 2. Score it using both metrics
    semantic_score = calculate_semantic_score(answer, gt)
    jaccard_score = calculate_jaccard(answer, gt)
    
    results.append({
        "Question": q,
        "Ground Truth": gt,
        "TISD Answer": answer,
        "Semantic Score": round(semantic_score, 4),
        "Old Jaccard Score": round(jaccard_score, 4),
        "Latency (sec)": round(latency, 2)
    })

# Convert to DataFrame
df_eval = pd.DataFrame(results)

In [ ]:
# Cell 4
avg_semantic = df_eval["Semantic Score"].mean()
avg_jaccard = df_eval["Old Jaccard Score"].mean()

display(HTML(f"""
<div style="background-color: #f0f4f8; padding: 20px; border-radius: 10px; border-left: 10px solid #2196F3;">
    <h2 style="margin-top: 0;">📊 Evaluation Revolution Report</h2>
    <p style="font-size: 18px;"><b>Average Semantic Accuracy: <span style="color: green;">{avg_semantic:.2%}</span></b></p>
    <p style="font-size: 14px; color: gray;">Old Jaccard Overlap: {avg_jaccard:.2%}</p>
    <p style="font-size: 16px;"><i>The semantic score reflects how well the AI understood the <b>meaning</b>, 
    ignoring the exact word choice.</i></p>
</div>
"""))

# Display the full table
display(df_eval.sort_values(by="Semantic Score", ascending=False))

In [ ]:
# Cell 5
# Pick a random successful case where Jaccard was low but Semantic was high
diff_df = df_eval[df_eval["Semantic Score"] > 0.7]
if not diff_df.empty:
    sample = diff_df.iloc[0]
    print("=== WHY SEMANTIC IS BETTER (SAMPLE) ===")
    print(f"Q: {sample['Question']}")
    print(f"GT: {sample['Ground Truth']}")
    print(f"AI: {sample['TISD Answer']}")
    print(f"Jaccard (Words): {sample['Old Jaccard Score']:.2%}")
    print(f"Semantic (Meaning): {sample['Semantic Score']:.2%}")